In [4]:
# ============================
# CHUNK 0R — Recovery de estado para NUCmer N2/N3
# ============================
import os, re
import pandas as pd
import numpy as np
from pathlib import Path
import gzip

# --- Paths base ---
DATA_ROOT   = Path("/home/nacho/HDD16/Nacho/RepliCOOC")
FASTA_DIR   = DATA_ROOT / "data" / "raw" / "fasta"
DEREP_DIR   = DATA_ROOT / "derep"
LOUVAIN_DIR = DEREP_DIR / "louvain_by_level_from_table_including_isolates"
OUTDIR      = DEREP_DIR / "containment_mm2"
NUC_DIR     = OUTDIR / "nucmer"

OUTDIR.mkdir(parents=True, exist_ok=True)
NUC_DIR.mkdir(parents=True, exist_ok=True)

REP_BY_LVL_TSV = LOUVAIN_DIR / "representatives_by_cluster_level.tsv"
LEVELS_ORDER   = [1,2,3,4]

# --- Parámetros (puedes reajustar) ---
STRICT_COV = 0.98
STRICT_PID = 0.90
LEN_TOL    = 0.005
MIN_ALN_BP = 100

# --- Carga tabla de representantes/levels ---
rep = pd.read_csv(REP_BY_LVL_TSV, sep="\t", dtype=str)
rep.columns = [c.strip() for c in rep.columns]

# Normaliza nombres de columnas -> plasmid_id / level
id_candidates    = {"plasmid_id","nuccore_acc","id","seqid","sequence_id","representative","rep","rep_id"}
level_candidates = {"level","cluster_level","louvain_level"}

id_col = next((c for c in rep.columns if c.lower() in id_candidates), None)
lvl_col = next((c for c in rep.columns if c.lower() in level_candidates), None)
assert id_col and lvl_col, f"No encuentro columnas ID/nivel en {REP_BY_LVL_TSV} -> {rep.columns}"

rep = rep.rename(columns={id_col:"plasmid_id", lvl_col:"level"}).copy()
rep["level"] = rep["level"].astype(str).str.extract(r"(\d+)").astype(int)

# --- Indexa FASTA: id -> path ---
valid_exts = (".fna", ".fa", ".fasta", ".fna.gz", ".fa.gz", ".fasta.gz")
def infer_id_from_filename(p: Path) -> str:
    s = p.name
    if s.endswith(".gz"): s = s[:-3]
    for ext in (".fna", ".fa", ".fasta"):
        if s.endswith(ext):
            s = s[:-len(ext)]
            break
    return s

fa_index = {}
for p in FASTA_DIR.iterdir():
    if p.is_file() and any(p.name.endswith(ext) for ext in valid_exts):
        fa_index[infer_id_from_filename(p)] = p.resolve()

rep["fasta"] = rep["plasmid_id"].map(fa_index)
missing = rep["fasta"].isna().sum()
if missing:
    print(f"⚠️ Faltan {missing} FASTA; muestra 5:")
    print(rep.loc[rep["fasta"].isna(), "plasmid_id"].head())

# --- Workplans (L vs >L) por si los necesitas de nuevo ---
workplans = []
for i, L in enumerate(LEVELS_ORDER):
    higher = LEVELS_ORDER[i+1:]
    if not higher:
        continue
    q = rep.query("level == @L").dropna(subset=["fasta"]).copy()
    t = rep.query("level >  @L").dropna(subset=["fasta"]).copy()
    if len(q) and len(t):
        workplans.append(dict(query_level=L, target_levels=higher, queries_df=q, targets_df=t))
print(f"[i] Planes disponibles: {len(workplans)}")

# --- Helpers N2: indexar headers y longitudes (contig y total) ---
def iter_fasta_records(path: Path):
    opener = gzip.open if str(path).endswith(".gz") else open
    with opener(path, "rt", errors="ignore") as fh:
        name, buf_len = None, 0
        for line in fh:
            if line.startswith(">"):
                if name is not None:
                    yield name, buf_len
                name = line[1:].strip().split()[0]
                buf_len = 0
            else:
                buf_len += len(line.strip())
        if name is not None:
            yield name, buf_len

SEQNAME_TO_ID = {}
CONTIG_LEN    = {}
PLASMID_LEN   = {}
for pid, fap in fa_index.items():
    total = 0
    for h, L in iter_fasta_records(Path(fap)):
        SEQNAME_TO_ID[h] = pid
        CONTIG_LEN[h]    = int(L)
        total           += int(L)
    PLASMID_LEN[pid] = int(total)

def _seq_to_id(name: str) -> str:
    return SEQNAME_TO_ID.get(name, name)

def merge_intervals(intervals):
    ints = [(int(a), int(b)) for a,b in intervals if int(b) > int(a)]
    if not ints:
        return 0
    ints.sort()
    merged = []
    cs, ce = ints[0]
    for s,e in ints[1:]:
        if s <= ce:
            ce = max(ce, e)
        else:
            merged.append((cs, ce))
            cs, ce = s, e
    merged.append((cs, ce))
    return sum(e - s for s, e in merged)

# --- Mapas útiles para N2/N3 ---
lv_map = rep.set_index("plasmid_id")["level"].to_dict()

print("[✓] Recovery listo. Puedes ejecutar directamente el CHUNK N2 y luego N3.")


[i] Planes disponibles: 3
[✓] Recovery listo. Puedes ejecutar directamente el CHUNK N2 y luego N3.


In [ ]:
# ============================
# CHUNK N1 — preparar y lanzar NUCmer por plan
# ============================
from pathlib import Path

# Reusa: OUTDIR, workplans ya definidos
NUC_DIR = OUTDIR / "nucmer"
NUC_DIR.mkdir(parents=True, exist_ok=True)

NUCMER_BIN = "nucmer"        # de MUMmer4
SHOWCOORDS = "show-coords"   # de MUMmer4

def write_fasta_of_list(paths, out_fa: Path):
    if out_fa.exists() and out_fa.stat().st_size > 0:
        return out_fa
    with out_fa.open("wb") as w:
        for p in paths:
            with open(p, "rb") as r:
                w.write(r.read())
    return out_fa

script = NUC_DIR / "run_nucmer_plans.sh"
lines = [
    "#!/usr/bin/env bash",
    "set -euo pipefail",
    'echo "[i] Running NUCmer (MUMmer4) with 30 threads"',
    ""
]

made = []
for wp in workplans:
    L = wp["query_level"]
    q_fas = list(map(Path, wp["queries_df"]["fasta"]))
    t_fas = list(map(Path, wp["targets_df"]["fasta"]))

    q_concat = NUC_DIR / f"level{L}_queries.fasta"
    t_concat = NUC_DIR / f"level_gt{L}_targets.fasta"
    write_fasta_of_list(q_fas, q_concat)
    write_fasta_of_list(t_fas, t_concat)

    prefix   = NUC_DIR / f"level{L}_vs_gt{L}"
    delta    = f"{prefix}.delta"
    coords   = f"{prefix}.coords.tsv"

    # --maxmatch = sensible para detección de contención; ajustable a --mum/--mumreference si quisieras
    # -t 30 usa 30 hilos
    lines += [
        f'echo "[i] L{L}: nucmer --maxmatch -t 30 -p {prefix} {t_concat} {q_concat}"',
        f'{NUCMER_BIN} --maxmatch -t 30 -p "{prefix}" "{t_concat}" "{q_concat}"',
        f'echo "[i] L{L}: show-coords -THrcl {delta} > {coords}"',
        f'{SHOWCOORDS} -THrcl "{delta}" > "{coords}"',
        f'echo "[✓] L{L} -> {coords}"',
        ""
    ]
    made.append((L, q_concat, t_concat, prefix))

with script.open("w") as f:
    f.write("\n".join(lines))
script.chmod(0o755)

print("[i] Script listo:", script)
for L, qfa, tfa, pref in made:
    print(f"  L{L}: queries={qfa.name}  targets={tfa.name}  prefix={pref.name}")

print("\nLanza en terminal:")
print(f"  bash {script}")


In [5]:
# ============================================
# N2 — leer *.cov90.tsv usando LONGITUD REAL desde TSV
#      añadir longitud alineada y quedarnos solo con n_hits == 1
# ============================================
import pandas as pd
import numpy as np
from pathlib import Path

NUC_DIR = OUTDIR / "nucmer"

STRICT_COV = 0.90
STRICT_PID = 0.90

# ---------- cargar niveles ----------
if "lv_map" not in globals():
    lv_map = rep.set_index("plasmid_id")["level"].to_dict()

# ---------- cargar longitudes reales desde TSV ----------
LEN_TSV = OUTDIR / "plasmid_lengths_from_fasta.tsv"
assert LEN_TSV.exists(), f"No existe TSV de longitudes: {LEN_TSV}"

len_df = pd.read_csv(LEN_TSV, sep="\t")
PLASMID_LEN = dict(zip(len_df["plasmid_id"], len_df["length_bp"]))

print(f"[i] PLASMID_LEN cargado: {len(PLASMID_LEN)} plasmidos.")

# ----------------------------------------------------------------------
# PROCESO
# ----------------------------------------------------------------------
all_edges = []

for cov_path in sorted(NUC_DIR.glob("level*_vs_gt*.cov90.tsv")):
    if cov_path.stat().st_size == 0:
        print(f"[i] Vacío: {cov_path}")
        continue

    print(f"[i] Leyendo {cov_path}")
    df = pd.read_csv(cov_path, sep=r"\s+", header=None)

    if df.shape[1] != 14:
        raise RuntimeError(f"{cov_path} tiene {df.shape[1]} columnas, esperaba 14.")

    df.columns = [
        "s1","e1","s2","e2","len1","len2","idy",
        "lenR","lenQ","covR","covQ","ref","qry","cov_small"
    ]

    # ---------- añadir longitudes reales ----------
    df["ref_len_real"] = df["ref"].map(PLASMID_LEN)
    df["qry_len_real"] = df["qry"].map(PLASMID_LEN)

    before = len(df)
    df = df.dropna(subset=["ref_len_real","qry_len_real"]).copy()
    if len(df) < before:
        print(f"[i] {before-len(df)} alineos descartados por falta de longitud real.")

    if df.empty:
        print(f"⚠️ {cov_path.name}: vacío tras asignar longitudes reales.")
        continue

    # ---------- determinar pequeño y grande con longitud REAL ----------
    df["small_len"] = np.where(df["ref_len_real"] <= df["qry_len_real"],
                               df["ref_len_real"], df["qry_len_real"])
    df["big_len"]   = np.where(df["ref_len_real"] <= df["qry_len_real"],
                               df["qry_len_real"], df["ref_len_real"])

    df["small_id"] = np.where(df["ref_len_real"] <= df["qry_len_real"],
                               df["ref"], df["qry"])
    df["big_id"]   = np.where(df["ref_len_real"] <= df["qry_len_real"],
                               df["qry"], df["ref"])

    # ---------- longitud alineada en el pequeño ----------
    df["aligned_small_bp"] = np.where(
        df["ref_len_real"] <= df["qry_len_real"],
        df["len1"],
        df["len2"]
    )

    # ---------- métricas ----------
    df["coverage"]     = df["aligned_small_bp"] / df["small_len"]
    df["pid_weighted"] = df["idy"] / 100.0

    # ---------- filtro cobertura/identidad ----------
    df_pass = df.query("coverage >= @STRICT_COV and pid_weighted >= @STRICT_PID").copy()
    if df_pass.empty:
        print(f"⚠️ {cov_path.name}: nada pasa STRICT_COV/STRICT_PID.")
        continue

    # ---------- agregación por par pequeño→grande ----------
    edges = (
        df_pass
        .groupby(["small_id","big_id"], as_index=False)
        .agg(
            coverage       = ("coverage","max"),
            pid_weighted   = ("pid_weighted","max"),
            n_hits         = ("small_id","size"),
            aligned_bp_sum = ("aligned_small_bp","sum"),
            small_len      = ("small_len","first"),
            big_len        = ("big_len","first")
        )
    )

    # ---------- solo casos con una única copia ----------
    edges = edges.query("n_hits == 1").copy()
    if edges.empty:
        print(f"⚠️ {cov_path.name}: sin n_hits==1.")
        continue

    # ---------- añadir niveles ----------
    edges["query_id"]   = edges["small_id"]
    edges["target_id"]  = edges["big_id"]
    edges["query_level"]  = edges["query_id"].map(lv_map)
    edges["target_level"] = edges["target_id"].map(lv_map)
    edges["source_file"]  = cov_path.name

    edges = edges[
        ["query_id","target_id","coverage","pid_weighted",
         "aligned_bp_sum","small_len","big_len","n_hits",
         "query_level","target_level","source_file"]
    ]

    print(f"[✓] {cov_path.name}: edges = {len(edges)}")
    all_edges.append(edges)

# ----------------------------------------------------------------------
# Consolidado final
# ----------------------------------------------------------------------
if all_edges:
    results_pass_nuc = pd.concat(all_edges, ignore_index=True)
else:
    results_pass_nuc = pd.DataFrame(columns=[
        "query_id","target_id","coverage","pid_weighted",
        "aligned_bp_sum","small_len","big_len","n_hits",
        "query_level","target_level","source_file"
    ])

out_tsv = NUC_DIR / "containment_NUC_from_cov90_PASS_singlehit.tsv"
results_pass_nuc.to_csv(out_tsv, sep="\t", index=False)

print(f"[i] Consolidado -> {out_tsv}  |  N={len(results_pass_nuc)}")
display(results_pass_nuc.head())


[i] PLASMID_LEN cargado: 72556 plasmidos.
[i] Leyendo /home/nacho/HDD16/Nacho/RepliCOOC/derep/containment_mm2/nucmer/level1_vs_gt1.cov90.tsv
[✓] level1_vs_gt1.cov90.tsv: edges = 1353
[i] Leyendo /home/nacho/HDD16/Nacho/RepliCOOC/derep/containment_mm2/nucmer/level2_vs_gt2.cov90.tsv
[✓] level2_vs_gt2.cov90.tsv: edges = 310
[i] Leyendo /home/nacho/HDD16/Nacho/RepliCOOC/derep/containment_mm2/nucmer/level3_vs_gt3.cov90.tsv
[✓] level3_vs_gt3.cov90.tsv: edges = 18
[i] Consolidado -> /home/nacho/HDD16/Nacho/RepliCOOC/derep/containment_mm2/nucmer/containment_NUC_from_cov90_PASS_singlehit.tsv  |  N=1681


,query_id,target_id,coverage,pid_weighted,aligned_bp_sum,small_len,big_len,n_hits,query_level,target_level,source_file
0,AP024697.1,NZ_CP068038.1,0.927230,0.9925,5186,5593,5597,1,1,2,level1_vs_gt1.cov90.tsv
1,AP026493.1,NZ_CP116156.1,0.915297,0.9989,5403,5903,5903,1,2,1,level1_vs_gt1.cov90.tsv
2,AP026592.1,NZ_CP017798.1,0.910591,0.9814,1986,2181,171477,1,1,2,level1_vs_gt1.cov90.tsv
3,AP026592.1,NZ_CP018827.1,0.910591,0.9814,1986,2181,259837,1,1,3,level1_vs_gt1.cov90.tsv
4,AP026592.1,NZ_CP023800.1,0.910591,0.9814,1986,2181,202540,1,1,3,level1_vs_gt1.cov90.tsv


In [6]:
import pandas as pd
from pathlib import Path

NUC_DIR = OUTDIR / "nucmer"
inp = NUC_DIR / "containment_NUC_from_cov90_PASS_singlehit.tsv"
df = pd.read_csv(inp, sep="\t")

print("[i] Entradas originales:", len(df))

# --- Filtro 1: solo nivel bajo -> nivel alto ---
df = df.query("query_level < target_level").copy()

# --- Filtro 2: pequeño estrictamente menor que grande ---
df = df.query("small_len < big_len").copy()

print("[i] Tras filtros nivel↑ y small<big:", len(df))

OUT = NUC_DIR / "containment_NUC_levelup_strictlen.tsv"
df.to_csv(OUT, sep="\t", index=False)

print(f"[✓] Guardado: {OUT}")
display(df.head())


[i] Entradas originales: 1681
[i] Tras filtros nivel↑ y small<big: 1617
[✓] Guardado: /home/nacho/HDD16/Nacho/RepliCOOC/derep/containment_mm2/nucmer/containment_NUC_levelup_strictlen.tsv


,query_id,target_id,coverage,pid_weighted,aligned_bp_sum,small_len,big_len,n_hits,query_level,target_level,source_file
0,AP024697.1,NZ_CP068038.1,0.927230,0.9925,5186,5593,5597,1,1,2,level1_vs_gt1.cov90.tsv
2,AP026592.1,NZ_CP017798.1,0.910591,0.9814,1986,2181,171477,1,1,2,level1_vs_gt1.cov90.tsv
3,AP026592.1,NZ_CP018827.1,0.910591,0.9814,1986,2181,259837,1,1,3,level1_vs_gt1.cov90.tsv
4,AP026592.1,NZ_CP023800.1,0.910591,0.9814,1986,2181,202540,1,1,3,level1_vs_gt1.cov90.tsv
5,AP026592.1,NZ_CP096212.1,0.916094,0.9805,1998,2181,291523,1,1,2,level1_vs_gt1.cov90.tsv


In [7]:
# ============================================
# N3 — Grafo y resúmenes (solo nivel↑, small<big)
#      tamaño nodo = tamaño del plásmido
# ============================================
import pandas as pd
import numpy as np
from pathlib import Path

NUC_DIR = OUTDIR / "nucmer"
GDIR    = NUC_DIR / "graph_cov90_noclose"
GDIR.mkdir(parents=True, exist_ok=True)

# -------- cargar edges ya filtrados por contención --------
pass_path = NUC_DIR / "containment_NUC_levelup_strictlen.tsv"
assert pass_path.exists() and pass_path.stat().st_size > 0, f"Falta {pass_path}"

results_pass_nuc = pd.read_csv(pass_path, sep="\t")

# copia y FILTRO FINAL: solo edges que SUBEN de nivel y small<big
edges = results_pass_nuc.copy()
edges = edges.query("target_level > query_level and small_len < big_len").copy()

edges.to_csv(GDIR / "edges_observed.tsv", sep="\t", index=False)
print(f"[i] Aristas observadas (solo nivel↑ y small<big): {len(edges)}")

if edges.empty:
    print("⚠️ No hay aristas tras el filtro, paro aquí.")
else:
    # -------- nodos --------
    nodes = pd.DataFrame({
        "plasmid_id": pd.unique(
            pd.concat([edges["query_id"], edges["target_id"]], ignore_index=True)
        )
    })
    lv_map = rep.set_index("plasmid_id")["level"].to_dict()
    nodes["level"] = nodes["plasmid_id"].map(lv_map)

    # -------- longitud del plásmido (para tamaño de nodo) --------
    # 1) si tenemos PLASMID_LEN en memoria, úsalo
    if "PLASMID_LEN" in globals():
        nodes["length_bp"] = nodes["plasmid_id"].map(PLASMID_LEN).astype("float")
    else:
        # 2) si no, inferir de small_len / big_len en edges
        len_sources = []
        if "small_len" in edges.columns:
            len_sources.append(
                edges[["query_id","small_len"]]
                    .rename(columns={"query_id":"plasmid_id","small_len":"length_bp"})
            )
            len_sources.append(
                edges[["target_id","small_len"]]
                    .rename(columns={"target_id":"plasmid_id","small_len":"length_bp"})
            )
        if "big_len" in edges.columns:
            len_sources.append(
                edges[["query_id","big_len"]]
                    .rename(columns={"query_id":"plasmid_id","big_len":"length_bp"})
            )
            len_sources.append(
                edges[["target_id","big_len"]]
                    .rename(columns={"target_id":"plasmid_id","big_len":"length_bp"})
            )

        if len_sources:
            len_df = (pd.concat(len_sources, ignore_index=True)
                        .dropna()
                        .drop_duplicates(subset=["plasmid_id"]))
            len_map = dict(zip(len_df["plasmid_id"], len_df["length_bp"]))
            nodes["length_bp"] = nodes["plasmid_id"].map(len_map).astype("float")
        else:
            nodes["length_bp"] = np.nan

    nodes.to_csv(GDIR / "nodes.tsv", sep="\t", index=False)

    # -------- resúmenes por nivel del pequeño (query) --------
    summary_levels = (
        edges.groupby("query_level", dropna=False)
             .agg(
                 n_pairs=("query_id","size"),
                 mean_cov=("coverage","mean"),
                 median_cov=("coverage","median"),
                 mean_pid=("pid_weighted","mean"),
                 median_pid=("pid_weighted","median")
             )
             .reset_index()
    )
    summary_levels.to_csv(GDIR / "summary_by_query_level.tsv", sep="\t", index=False)

    # -------- flujo nivel->nivel --------
    flow = edges.groupby(["query_level","target_level"]).size().reset_index(name="n_edges")
    flow_pivot = (
        flow.pivot(index="query_level", columns="target_level",
                   values="n_edges").fillna(0).astype(int)
    )
    flow.to_csv(GDIR / "flows_level_to_level.tsv", sep="\t", index=False)

    # -------- mejor contención por query --------
    best_by_query = (
        edges.sort_values(["query_id","coverage","pid_weighted"],
                          ascending=[True, False, False])
             .groupby("query_id", as_index=False)
             .head(1)
    )
    best_by_query.to_csv(GDIR / "edges_best_per_query.tsv", sep="\t", index=False)

    display(edges.head())
    display(summary_levels)
    display(flow_pivot)

    # -------- grafo pyvis --------
    try:
        from pyvis.network import Network
        import math

        net = Network(height="900px", width="100%", bgcolor="#111111",
                      font_color="#EEEEEE", notebook=False, directed=True)
        net.barnes_hut(gravity=-30000, central_gravity=0.15,
                       spring_length=160, spring_strength=0.01, damping=0.9)

        lvl_colors = {1:"#8dd3c7", 2:"#ffffb3", 3:"#bebada", 4:"#fb8072",
                      5:"#80b1d3", 6:"#fdb462", 7:"#b3de69"}

        # nodos
        for _, r in nodes.iterrows():
            pid = r["plasmid_id"]
            lvl = int(r["level"]) if pd.notna(r["level"]) else None
            color = lvl_colors.get(lvl, "#cccccc")

            length = r.get("length_bp", np.nan)
            if pd.notna(length) and length > 0:
                size = max(8, min(40, 8 + 4 * math.log10(length)))
                title = f"{pid}<br>level={lvl}<br>length={int(length)} bp"
            else:
                size = 10
                title = f"{pid}<br>level={lvl}"

            net.add_node(pid, label=pid, color=color, title=title, size=size)

        # aristas
        for _, e in edges.iterrows():
            q, t = e["query_id"], e["target_id"]
            cov  = float(e["coverage"])
            pidw = float(e["pid_weighted"])
            title = (
                f"coverage_small={cov:.3f} | "
                f"pid={pidw:.3f} | hits={int(e['n_hits'])}"
            )
            width = 1 + 4*cov
            net.add_edge(q, t, value=cov, title=title, width=width, arrows="to")

        html_path = GDIR / "containment_network_cov90.html"
        net.write_html(str(html_path), open_browser=False)
        print(f"[✓] Grafo guardado: {html_path}")
    except Exception as e:
        print("⚠️ No se pudo generar el grafo (pyvis). Error:")
        print(e)


[i] Aristas observadas (solo nivel↑ y small<big): 1617


,query_id,target_id,coverage,pid_weighted,aligned_bp_sum,small_len,big_len,n_hits,query_level,target_level,source_file
0,AP024697.1,NZ_CP068038.1,0.927230,0.9925,5186,5593,5597,1,1,2,level1_vs_gt1.cov90.tsv
1,AP026592.1,NZ_CP017798.1,0.910591,0.9814,1986,2181,171477,1,1,2,level1_vs_gt1.cov90.tsv
2,AP026592.1,NZ_CP018827.1,0.910591,0.9814,1986,2181,259837,1,1,3,level1_vs_gt1.cov90.tsv
3,AP026592.1,NZ_CP023800.1,0.910591,0.9814,1986,2181,202540,1,1,3,level1_vs_gt1.cov90.tsv
4,AP026592.1,NZ_CP096212.1,0.916094,0.9805,1998,2181,291523,1,1,2,level1_vs_gt1.cov90.tsv


,query_level,n_pairs,mean_cov,median_cov,mean_pid,median_pid
0,1,1294,0.982726,1.000000,0.990402,0.9993
1,2,308,0.982184,1.000000,0.995516,0.9990
2,3,15,0.954553,0.955139,0.997027,0.9989


target_level,2,3,4,5,6,7,8
query_level,,,,,,,
1,523,409,225,106,27,2,2
2,0,190,74,38,6,0,0
3,0,0,9,5,1,0,0


[✓] Grafo guardado: /home/nacho/HDD16/Nacho/RepliCOOC/derep/containment_mm2/nucmer/graph_cov90_noclose/containment_network_cov90.html


In [ ]:
# ============================================
# N4 — Extraer cadenas L1 -> L2 / L3 / ... (sin ciclos)
# ============================================
import pandas as pd
from collections import defaultdict

NUC_DIR = OUTDIR / "nucmer"

# -------- Cargar edges filtrados --------
pass_path = NUC_DIR / "containment_NUC_levelup_strictlen.tsv"
assert pass_path.exists() and pass_path.stat().st_size > 0, f"Falta {pass_path}"

results_pass_nuc = pd.read_csv(pass_path, sep="\t")
edges = results_pass_nuc.copy()

# Asegura que tenemos lv_map
if 'lv_map' not in globals():
    lv_map = rep.set_index("plasmid_id")["level"].to_dict()

# Recalcular niveles por si acaso
edges["query_level"]  = edges["query_id"].map(lv_map)
edges["target_level"] = edges["target_id"].map(lv_map)

# Filtro de seguridad: solo aristas que suben de nivel y small<big
edges = edges.query("target_level > query_level and small_len < big_len").copy()

print(f"[i] Aristas usadas para cadenas (nivel↑ & small<big): {len(edges)}")

if edges.empty:
    print("⚠️ No hay aristas válidas para construir cadenas, paro aquí.")
else:
    # -------- Adyacencia dirigido: pequeño -> grande --------
    adj = defaultdict(list)
    for _, r in edges.iterrows():
        adj[r["query_id"]].append(r["target_id"])

    # Ordena vecinos por nivel y luego por id para reproducibilidad
    for k in adj:
        adj[k] = sorted(adj[k], key=lambda pid: (lv_map.get(pid, 999), pid))

    chains = []

    def dfs(path):
        """
        path: lista de plasmid_id desde el L1 inicial hasta el nodo actual.
        Extendemos mientras haya edges a niveles superiores; cuando no se puede
        extender más, guardamos la cadena (si tiene longitud >=2 y empieza en L1).
        """
        last = path[-1]
        last_lvl = lv_map.get(last, None)
        extended = False

        for nxt in adj.get(last, []):
            if nxt in path:
                continue  # evita ciclos por si acaso
            nxt_lvl = lv_map.get(nxt, None)
            if nxt_lvl is None or (last_lvl is not None and nxt_lvl <= last_lvl):
                continue
            dfs(path + [nxt])
            extended = True

        # Si no se ha podido extender, guardamos la cadena terminal
        if not extended and len(path) > 1 and lv_map.get(path[0], None) == 1:
            chains.append(path)

    # -------- Lanzar DFS desde todos los plasmidos de nivel 1 con salida --------
    starts = sorted(
        {pid for pid, lvl in lv_map.items() if lvl == 1 and pid in adj},
        key=lambda pid: pid
    )

    print(f"[i] Nodos de inicio (L1 con salidas): {len(starts)}")

    for s in starts:
        dfs([s])

    print(f"[i] Cadenas L1->... encontradas: {len(chains)}")

    # -------- Pasar a tabla larga --------
    rows = []
    for cid, path in enumerate(chains, start=1):
        for pos, pid in enumerate(path, start=1):
            lvl = lv_map.get(pid, None)
            rows.append({
                "chain_id": f"chain_{cid:04d}",
                "order_in_chain": pos,
                "plasmid_id": pid,
                "level": lvl
            })

    chains_df = (
        pd.DataFrame(rows)
          .sort_values(["chain_id","order_in_chain"])
          .reset_index(drop=True)
    )

    CHAINS_TSV = NUC_DIR / "chains_L1_up.tsv"
    chains_df.to_csv(CHAINS_TSV, sep="\t", index=False)

    print(f"[i] Guardado listado de cadenas en: {CHAINS_TSV}")
    display(chains_df.head(20))


In [ ]:
# ============================================
# N5 — Copiar FASTA de plasmidos en cadenas para host-range
# ============================================
import os
import shutil
from pathlib import Path
import pandas as pd

NUC_DIR = OUTDIR / "nucmer"
CHAINS_TSV = NUC_DIR / "chains_L1_up.tsv"
OUT_DIR = NUC_DIR / "hostrange_fastas"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 1) Cargar cadenas
chains_df = pd.read_csv(CHAINS_TSV, sep="\t")

# 2) Lista de plasmidos únicos en cadenas
plasmids_in_chains = sorted(chains_df["plasmid_id"].unique())
print(f"[i] Plásmidos distintos en cadenas: {len(plasmids_in_chains)}")

# 3) Necesitamos fa_index (plasmid_id -> ruta fasta)
assert "fa_index" in globals(), "No encuentro fa_index en memoria (plasmid_id -> path FASTA)."

meta_rows = []
missing = []

for pid in plasmids_in_chains:
    src = fa_index.get(pid)
    if src is None:
        print(f"⚠️ No encuentro FASTA para {pid}, lo omito.")
        missing.append(pid)
        continue

    src = Path(src)
    # Nombre de salida: mantenemos el nombre del fichero original
    dst = OUT_DIR / src.name

    # Si ya existe, lo sobrescribimos para evitar residuos viejos
    if dst.exists():
        dst.unlink()

    shutil.copy2(src, dst)

    meta_rows.append({
        "plasmid_id": pid,
        "fasta_path": str(dst),
    })

# 4) Metadata TSV para referencia
meta_df = pd.DataFrame(meta_rows).sort_values("plasmid_id")
META_TSV = OUT_DIR / "hostrange_metadata.tsv"
meta_df.to_csv(META_TSV, sep="\t", index=False)

print(f"[✓] Copiados {len(meta_df)} FASTA a: {OUT_DIR}")
print(f"[i] Metadata en: {META_TSV}")
if missing:
    print(f"⚠️ Plásmidos sin FASTA en fa_index: {len(missing)} (ejemplo: {missing[:5]})")

display(meta_df.head(20))


In [ ]:
# ============================================
# HR0 — Cargar HostRange_Family.tsv de forma robusta
# ============================================
import pandas as pd
import re

HR_FAM_PATH = Path("/home/nacho/HRPredict/HostRange_Family.tsv")

hr_rows = []

with open(HR_FAM_PATH, "r") as fh:
    for line in fh:
        line = line.rstrip("\n")
        if not line.strip():
            continue

        parts = line.split("\t")
        pid = parts[0].strip()

        # Caso especial: sin predicción
        if len(parts) == 1 or parts[1].startswith("No "):
            continue

        for field in parts[1:]:
            field = field.strip()
            if not field:
                continue

            # patrón:   f__Enterobacteriaceae(1.23)
            m = re.match(r"([a-z]__[^()]+)\((-?[0-9.]+)\)", field)
            if not m:
                continue

            taxon = m.group(1)
            val = float(m.group(2))

            hr_rows.append((pid, taxon, val))

hr_fam_long = pd.DataFrame(hr_rows, columns=["plasmid_id", "taxon", "decision"])

print(f"[i] Filas parseadas: {len(hr_fam_long)}")
display(hr_fam_long.head(10))


In [8]:
# ============================================
# HR1 — generar métricas por plasmidio
# ============================================
HR_THRESH = 0.1  # decision > 0 significa "fam. potencial hospedadora"

hr_score_fam = (
    hr_fam_long
    .query("decision > @HR_THRESH")
    .groupby("plasmid_id", as_index=False)
    .agg(
        host_range_n = ("taxon", "nunique"),
        host_range_sum = ("decision", "sum"),
    )
)

print(f"[i] Plásmidos con host_range_n > 0: {len(hr_score_fam)}")
display(hr_score_fam.head(10))
# ============================================
# HR2 — Unir cadenas con host range
# ============================================
CHAINS_TSV = NUC_DIR / "chains_L1_up.tsv"
chains = pd.read_csv(CHAINS_TSV, sep="\t")

chains_hr = chains.merge(
    hr_score_fam,
    on="plasmid_id",
    how="left"
)

# Rellenar plasmidos sin predicción con 0
chains_hr["host_range_n"] = chains_hr["host_range_n"].fillna(0)
chains_hr["host_range_sum"] = chains_hr["host_range_sum"].fillna(0)

display(chains_hr.head(10))
# ============================================
# HR3A — Comparar host range entre posiciones consecutivas en cada cadena
# ============================================
pairs = chains_hr.copy()

# Shift por cadena para comparar Lk → Lk+1
pairs = pairs.merge(
    chains_hr.rename(columns={
        "plasmid_id":"plasmid_id_next",
        "level":"level_next",
        "host_range_n":"host_range_n_next",
        "order_in_chain":"order_in_chain_next"
    }),
    on=["chain_id"],
    how="left"
)

# Solo pares consecutivos
pairs = pairs.query("order_in_chain_next == order_in_chain + 1").copy()

# Delta host range
pairs["delta_hr"] = pairs["host_range_n_next"] - pairs["host_range_n"]

display(pairs[[
    "chain_id","plasmid_id","plasmid_id_next",
    "host_range_n","host_range_n_next","delta_hr"
]].head(20))

print("[i] % casos con delta_hr > 0:", (pairs["delta_hr"] > 0).mean())
from scipy.stats import wilcoxon

valid = pairs.dropna(subset=["host_range_n","host_range_n_next"])

stat, pval = wilcoxon(
    valid["host_range_n"],
    valid["host_range_n_next"],
    alternative="less"   # esperamos host_range_n < host_range_n_next
)

print(f"[i] Wilcoxon: stat={stat}, pval={pval}")
# ============================================
# HR3C — correlación de host range vs orden en la cadena
# ============================================
stats_rows = []

for cid, sub in chains_hr.groupby("chain_id"):
    if sub["host_range_n"].nunique() == 1:
        continue
    corr = sub[["order_in_chain","host_range_n"]].corr().iloc[0,1]
    stats_rows.append({"chain_id":cid, "corr_order_vs_hr": corr})

chain_stats = pd.DataFrame(stats_rows)
display(chain_stats.head())

print("[i] % cadenas con correlación positiva:",
      (chain_stats["corr_order_vs_hr"] > 0).mean())


NameError: name 'hr_fam_long' is not defined

In [13]:
# ============================================================
# UNIONES EXACTAS DE PLÁSMIDOS USANDO CONTENCIÓN + REPLICONES
# + CONTROL DE LONGITUD (±10%)
# ============================================================

import pandas as pd
import itertools
from pathlib import Path

# ------------------------------------------------------------
# INPUT 1: edges (ya existente en el entorno)
# ------------------------------------------------------------
# Requiere columnas:
# query_id, target_id, coverage, pid_weighted,
# query_level, target_level
edges = edges.copy()

# ------------------------------------------------------------
# INPUT 2: replicones
# ------------------------------------------------------------
REPL_TSV = Path("/home/nacho/HDD16/Nacho/RepliCOOC/derep/plasmid_replicons.tsv")
assert REPL_TSV.exists(), f"No existe {REPL_TSV}"

replicons_df = pd.read_csv(REPL_TSV, sep="\t")

replicons_df["replicon_set"] = (
    replicons_df["replicon_types"]
    .astype(str)
    .str.split(",")
    .apply(lambda xs: frozenset(x.strip() for x in xs if x.strip()))
)

replicon_map = dict(
    zip(replicons_df["plasmid_id"], replicons_df["replicon_set"])
)

# ------------------------------------------------------------
# INPUT 3: longitudes desde nuccore.csv
# ------------------------------------------------------------
NUCCORE_CSV = Path(
    "/home/nacho/HDD16/Nacho/RepliCOOC/derep/cell_level/meta/nuccore.csv"
)
assert NUCCORE_CSV.exists(), f"No existe {NUCCORE_CSV}"

nuccore_df = pd.read_csv(NUCCORE_CSV)

# solo entradas plasmídicas
nuccore_df = nuccore_df[nuccore_df["NUCCORE_Genome"] == "plasmid"]

# mapa: plasmid_id (= NUCCORE_ACC) → longitud
length_map = dict(
    zip(nuccore_df["NUCCORE_ACC"], nuccore_df["NUCCORE_Length"])
)

# ------------------------------------------------------------
# RELACIONES DE CONTENCIÓN (para excluir N⊂K o K⊂N)
# ------------------------------------------------------------
containment_pairs = set(
    zip(edges["query_id"], edges["target_id"])
)

def contains(a, b):
    return (a, b) in containment_pairs

# ------------------------------------------------------------
# CANDIDATOS A COMPUESTOS (indegree ≥ 2)
# ------------------------------------------------------------
parents_by_target = (
    edges.groupby("target_id")["query_id"]
    .apply(set)
    .to_dict()
)

compound_candidates = {
    c: parents
    for c, parents in parents_by_target.items()
    if len(parents) >= 2
}

# ------------------------------------------------------------
# DETECCIÓN DE UNIONES EXACTAS
# ------------------------------------------------------------
results = []

for compound, parents in compound_candidates.items():

    R_comp = replicon_map.get(compound)
    if not R_comp or len(R_comp) < 2:
        continue

    L_comp = length_map.get(compound)
    if L_comp is None:
        continue

    valid_parents = [
        p for p in parents
        if p in replicon_map and replicon_map[p] and p in length_map
    ]

    for k in range(2, len(valid_parents) + 1):
        for subset in itertools.combinations(valid_parents, k):

            # --- independencia jerárquica ---
            if any(contains(a, b) for a, b in itertools.permutations(subset, 2)):
                continue

            # --- unión EXACTA de replicones ---
            union_reps = set().union(*(replicon_map[p] for p in subset))
            if union_reps != R_comp:
                continue

            # --- control de longitud (±10%) ---
            L_sum = sum(length_map[p] for p in subset)
            ratio = L_sum / L_comp
            if not (0.9 <= ratio <= 1.1):
                continue

            # --- métricas asociadas ---
            sub_edges = edges[
                (edges["query_id"].isin(subset)) &
                (edges["target_id"] == compound)
            ]

            results.append({
                "compound": compound,
                "components": ",".join(sorted(subset)),
                "n_components": len(subset),
                "replicon_types_compound": ",".join(sorted(R_comp)),
                "mean_coverage": sub_edges["coverage"].mean(),
                "min_coverage": sub_edges["coverage"].min(),
                "mean_pid": sub_edges["pid_weighted"].mean(),
                "levels_components": ",".join(
                    sorted(sub_edges["query_level"].astype(str).unique())
                ),
                "level_compound": sub_edges["target_level"].iloc[0],
                "length_compound": L_comp,
                "length_components_sum": L_sum,
                "length_ratio": ratio
            })

# ------------------------------------------------------------
# RESULTADO FINAL
# ------------------------------------------------------------
unions_df = (
    pd.DataFrame(results)
    .sort_values(["n_components", "compound"])
    .reset_index(drop=True)
)

unions_df


,compound,components,n_components,replicon_types_compound,mean_coverage,min_coverage,mean_pid,levels_components,level_compound,length_compound,length_components_sum,length_ratio
0,CP050286.1,"NZ_CP064260.1,NZ_CP095263.1",2,"IncFIA,IncFII,IncR",0.976139,0.952279,0.998350,2,3,103455,96249,0.930347
1,CP070405.1,"NZ_CP056078.1,NZ_CP148459.1",2,"Col(MG828),rep_cluster_2350",0.948079,0.926442,0.990950,1,2,5160,5141,0.996318
2,CP070405.1,"NZ_CP056078.1,NZ_CP142413.1",2,"Col(MG828),rep_cluster_2350",0.961783,0.953850,0.993550,1,2,5160,5149,0.997868
3,CP070405.1,"CP042626.1,NZ_CP148459.1",2,"Col(MG828),rep_cluster_2350",0.954195,0.926442,0.985800,1,2,5160,5140,0.996124
4,CP070405.1,"NZ_CP076700.1,NZ_CP148459.1",2,"Col(MG828),rep_cluster_2350",0.916649,0.906856,0.970650,1,2,5160,5135,0.995155
...,...,...,...,...,...,...,...,...,...,...,...,...
136,NZ_CP082050.1,"CP088717.1,NZ_CP045284.1,NZ_CP070154.1,NZ_CP14...",4,"Col156,rep_cluster_2131",0.992742,0.985479,0.985025,1,2,13179,14120,1.071401
137,NZ_CP082050.1,"CP088360.1,CP088717.1,NZ_CP070154.1,NZ_CP148389.1",4,"Col156,rep_cluster_2131",0.981576,0.940824,0.985975,1,2,13179,14123,1.071629
138,NZ_CP082050.1,"CP077352.1,CP088717.1,NZ_CP045284.1,NZ_CP070154.1",4,"Col156,rep_cluster_2131",0.996372,0.985488,0.987525,1,2,13179,11879,0.901358
139,NZ_CP082050.1,"CP077352.1,CP088360.1,CP088717.1,NZ_CP070154.1",4,"Col156,rep_cluster_2131",0.985206,0.940824,0.988475,1,2,13179,11882,0.901586


In [14]:
OUT = Path("/home/nacho/HDD16/Nacho/RepliCOOC/derep/union_plasmids_exact.tsv")
unions_df.to_csv(OUT, sep="\t", index=False)
OUT


PosixPath('/home/nacho/HDD16/Nacho/RepliCOOC/derep/union_plasmids_exact.tsv')